# 🎬 MimicMotion — Transferencia de Movimiento: Foto → Video

Este notebook te permite animar una **foto de un personaje** usando el movimiento de un **video de referencia**.  
Basado en [MimicMotion (Tencent)](https://github.com/tencent/MimicMotion).

## Flujo de trabajo
1. Instalar dependencias
2. Descargar modelos
3. Subir tu foto y video de referencia
4. Configurar parámetros
5. Generar el video animado

---
> **GPU requerida**: Activa la GPU en `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`

## Paso 0 — Verificar GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ GPU no detectada. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU')

## Paso 1 — Clonar repositorio e instalar dependencias

In [ ]:
import os

# Clonar MimicMotion
if not os.path.exists('/content/MimicMotion'):
    !git clone https://github.com/tencent/MimicMotion /content/MimicMotion
else:
    print('✅ Repositorio ya clonado')

os.chdir('/content/MimicMotion')
print('📁 Directorio de trabajo:', os.getcwd())

In [ ]:
# ── Parche 1: cached_download eliminado en huggingface_hub>=0.24 ─────────────
import huggingface_hub as _hfh
if not hasattr(_hfh, 'cached_download'):
    _hfh.cached_download = _hfh.hf_hub_download
    print('🔧 Patch huggingface_hub.cached_download aplicado')
print(f'✅ huggingface_hub {_hfh.__version__}')

# ── Parche 2: clear_device_cache añadido en accelerate>=0.26 ─────────────────
import accelerate.utils.memory as _aum
if not hasattr(_aum, 'clear_device_cache'):
    import torch as _torch
    def _clear_device_cache():
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
    _aum.clear_device_cache = _clear_device_cache
    import accelerate.utils as _au
    _au.clear_device_cache = _clear_device_cache
    print('🔧 Patch accelerate.clear_device_cache aplicado')
print(f'✅ accelerate {__import__("accelerate").__version__}')

# ── Dependencias con versiones mínimas necesarias ────────────────────────────
!pip install -q \
    "diffusers==0.27.2" \
    "transformers>=4.38,<5.0" \
    "accelerate>=0.24" \
    omegaconf \
    einops \
    "imageio[ffmpeg]" \
    onnxruntime-gpu \
    opencv-python-headless \
    Pillow \
    decord \
    pyyaml

print('✅ Dependencias instaladas')

# ── Re-aplicar ambos parches tras la instalación ─────────────────────────────
import sys
for mod in list(sys.modules.keys()):
    if any(k in mod for k in ('diffusers', 'huggingface_hub', 'accelerate', 'peft')):
        del sys.modules[mod]

import huggingface_hub as _hfh
if not hasattr(_hfh, 'cached_download'):
    _hfh.cached_download = _hfh.hf_hub_download

import accelerate.utils.memory as _aum
if not hasattr(_aum, 'clear_device_cache'):
    import torch as _torch
    def _clear_device_cache():
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
    _aum.clear_device_cache = _clear_device_cache
    import accelerate.utils as _au
    _au.clear_device_cache = _clear_device_cache

print('✅ Todos los parches activos — listo para inferencia')

## Paso 2 — Descargar modelos

> ⏳ La descarga puede tomar 5–15 minutos dependiendo de la conexión.

In [ ]:
import os

os.makedirs('/content/MimicMotion/models/DWPose', exist_ok=True)

# Modelos DWPose (detección de pose corporal)
dwpose_files = {
    'yolox_l.onnx':        'https://huggingface.co/yzd-v/DWPose/resolve/main/yolox_l.onnx',
    'dw-ll_ucoco_384.onnx':'https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.onnx',
}

for filename, url in dwpose_files.items():
    filepath = f'/content/MimicMotion/models/DWPose/{filename}'
    if not os.path.exists(filepath):
        print(f'⬇️  Descargando {filename}...')
        os.system(f'wget -q "{url}" -O "{filepath}"')
        print(f'✅ {filename} descargado')
    else:
        print(f'✅ {filename} ya existe')

# Checkpoint principal de MimicMotion (~3 GB)
mimic_path = '/content/MimicMotion/models/MimicMotion_1-1.pth'
if not os.path.exists(mimic_path):
    print('⬇️  Descargando MimicMotion_1-1.pth (~3 GB)...')
    # Usar os.system para que la variable Python se expanda correctamente
    os.system(
        f'wget -q https://huggingface.co/tencent/MimicMotion/resolve/main/MimicMotion_1-1.pth'
        f' -O "{mimic_path}"'
    )
    print('✅ MimicMotion_1-1.pth descargado')
else:
    print('✅ MimicMotion_1-1.pth ya existe')

print('\n📦 Modelos listos:')
!ls -lh /content/MimicMotion/models/DWPose/
!ls -lh /content/MimicMotion/models/*.pth

## Paso 3 — Subir archivos de entrada

Sube:
- **Foto del personaje** (`.jpg` / `.png`) — La persona que quieres animar
- **Video de referencia** (`.mp4`) — El video cuyo movimiento se transferirá

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('/content/inputs', exist_ok=True)

print('📸 Sube la FOTO del personaje (jpg/png):')
uploaded_img = files.upload()

ref_image_path = None
for fname in uploaded_img:
    dest = f'/content/inputs/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded_img[fname])
    ref_image_path = dest
    print(f'✅ Foto guardada en: {ref_image_path}')

In [ ]:
print('🎥 Sube el VIDEO DE REFERENCIA (mp4):')
uploaded_vid = files.upload()

ref_video_path = None
for fname in uploaded_vid:
    dest = f'/content/inputs/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded_vid[fname])
    ref_video_path = dest
    print(f'✅ Video guardado en: {ref_video_path}')

In [ ]:
# Verificar que los archivos existen
from PIL import Image
import IPython.display as display
import cv2

assert ref_image_path and os.path.exists(ref_image_path), '❌ No se encontró la foto. Ejecuta la celda anterior.'
assert ref_video_path and os.path.exists(ref_video_path), '❌ No se encontró el video. Ejecuta la celda anterior.'

# Mostrar la foto cargada
img = Image.open(ref_image_path)
print(f'Foto: {img.size[0]}x{img.size[1]} px')
display.display(img.resize((256, int(256 * img.size[1] / img.size[0]))))

# Info del video
cap = cv2.VideoCapture(ref_video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_vid = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {w}x{h} px | {total_frames} frames | {fps_vid:.1f} fps | {total_frames/fps_vid:.1f}s')

## Paso 4 — Configurar parámetros

Ajusta los valores según tus necesidades y los recursos disponibles.

In [ ]:
# ============================================================
#  PARÁMETROS DE GENERACIÓN — modifica aquí
# ============================================================

# Resolución de salida (alto en píxeles). El ancho se calcula automáticamente.
# Recomendado: 512 para T4 (16GB), 576 para A100
RESOLUTION = 512  # @param {type:"slider", min:256, max:576, step:64}

# Número de frames a generar (máx 72 con MimicMotion 1.1)
# Menos frames = más rápido y menos VRAM
NUM_FRAMES = 48  # @param {type:"slider", min:16, max:72, step:8}

# Stride de muestreo: cada cuántos frames del video de referencia se toma uno
# 1 = todos, 2 = uno de cada dos (video más rápido), etc.
SAMPLE_STRIDE = 2  # @param {type:"slider", min:1, max:4, step:1}

# Frames de solapamiento entre segmentos (suaviza transiciones)
FRAMES_OVERLAP = 6  # @param {type:"slider", min:2, max:12, step:2}

# Pasos de inferencia del modelo de difusión (más pasos = mejor calidad, más lento)
NUM_INFERENCE_STEPS = 25  # @param {type:"slider", min:10, max:50, step:5}

# Escala de guiado (guidance scale): cuánto sigue la pose de referencia
GUIDANCE_SCALE = 2.0  # @param {type:"number"}

# Semilla aleatoria (42 = reproducible, -1 = aleatoria)
SEED = 42  # @param {type:"integer"}

# FPS del video de salida
OUTPUT_FPS = 15  # @param {type:"slider", min:8, max:30, step:1}

# Aumentación de ruido (0 = desactivado, valores pequeños añaden variación)
NOISE_AUG_STRENGTH = 0.0  # @param {type:"number"}

# Usar float16 para ahorrar VRAM (recomendado True en T4)
USE_FLOAT16 = True  # @param {type:"boolean"}

print('⚙️  Configuración:')
print(f'  Resolución     : {RESOLUTION}px')
print(f'  Frames         : {NUM_FRAMES}')
print(f'  Sample stride  : {SAMPLE_STRIDE}')
print(f'  Overlap frames : {FRAMES_OVERLAP}')
print(f'  Pasos difusión : {NUM_INFERENCE_STEPS}')
print(f'  Guidance scale : {GUIDANCE_SCALE}')
print(f'  Semilla        : {SEED}')
print(f'  FPS salida     : {OUTPUT_FPS}')
print(f'  Float16        : {USE_FLOAT16}')

## Paso 5 — Generar configuración YAML e inferencia

> ⏳ La primera ejecución descarga el modelo SVD (~6GB). Puede tomar 10–20 minutos en total.

In [ ]:
import os
from huggingface_hub import login, whoami

# ── Opción A: pegar el token directamente (más rápido) ──────────────────────
HF_TOKEN = ""  # @param {type:"string"}
# ── Opción B: usar variable de entorno (si usas Colab Secrets) ──────────────
# HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)
    try:
        info = whoami()
        print(f'✅ Autenticado como: {info["name"]}')
    except Exception as e:
        print(f'⚠️  Token aceptado pero no se pudo verificar el usuario: {e}')
else:
    # Intentar login interactivo
    print('No se proporcionó token. Ejecutando login interactivo...')
    print('(También puedes pegar tu token en HF_TOKEN arriba y re-ejecutar)')
    login()

# Verificar acceso al modelo SVD
print('\n🔍 Verificando acceso al modelo SVD...')
try:
    from huggingface_hub import model_info
    model_info("stabilityai/stable-video-diffusion-img2vid-xt-1-1")
    print('✅ Acceso confirmado a stable-video-diffusion-img2vid-xt-1-1')
except Exception as e:
    print(f'❌ Sin acceso: {e}')
    print('\n👉 Ve a https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt-1-1')
    print('   y acepta los términos de uso antes de continuar.')

import os

MIMIC_DIR = '/content/MimicMotion'
os.chdir(MIMIC_DIR)
os.makedirs('outputs', exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:256'

# Propagar token HF al subproceso
_tok = os.environ.get('HF_TOKEN') or (HF_TOKEN.strip() if 'HF_TOKEN' in dir() and HF_TOKEN else '')
if _tok:
    os.environ['HF_TOKEN'] = _tok
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _tok

no_fp16_args = ", '--no_use_float16'" if not USE_FLOAT16 else ''

wrapper_code = f"""
import os, sys

# Fijar directorio de trabajo e importación del paquete mimicmotion
MIMIC_DIR = '/content/MimicMotion'
os.chdir(MIMIC_DIR)
if MIMIC_DIR not in sys.path:
    sys.path.insert(0, MIMIC_DIR)

# ── Parche 1: huggingface_hub.cached_download ────────────────────────────────
import huggingface_hub as _h
if not hasattr(_h, 'cached_download'):
    _h.cached_download = _h.hf_hub_download

# ── Parche 2: accelerate.utils.memory.clear_device_cache ────────────────────
import accelerate.utils.memory as _aum
if not hasattr(_aum, 'clear_device_cache'):
    import torch as _t
    def _cdc():
        if _t.cuda.is_available(): _t.cuda.empty_cache()
    _aum.clear_device_cache = _cdc
    import accelerate.utils as _au
    _au.clear_device_cache = _cdc

# ── Login HF si hay token ────────────────────────────────────────────────────
_tok = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if _tok:
    _h.login(token=_tok, add_to_git_credential=False)

# ── Ejecutar inferencia ──────────────────────────────────────────────────────
sys.argv = ['inference.py',
            '--inference_config', '{config_path}',
            '--output_dir', 'outputs/'{no_fp16_args}]

with open(os.path.join(MIMIC_DIR, 'inference.py')) as _f:
    exec(_f.read(), {{'__name__': '__main__', '__file__': os.path.join(MIMIC_DIR, 'inference.py')}})
"""

with open('/tmp/run_inference.py', 'w') as f:
    f.write(wrapper_code)

print('🚀 Ejecutando inferencia...')
print('─' * 60)
!PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:256 python /tmp/run_inference.py

In [ ]:
import yaml, os

os.chdir('/content/MimicMotion')
os.makedirs('outputs', exist_ok=True)

config = {
    'base_model_path': 'stabilityai/stable-video-diffusion-img2vid-xt-1-1',
    'ckpt_path': 'models/MimicMotion_1-1.pth',
    'test_cases': [
        {
            'ref_video_path': ref_video_path,
            'ref_image_path': ref_image_path,
            'num_frames': NUM_FRAMES,
            'resolution': RESOLUTION,
            'frames_overlap': FRAMES_OVERLAP,
            'num_inference_steps': NUM_INFERENCE_STEPS,
            'noise_aug_strength': NOISE_AUG_STRENGTH,
            'guidance_scale': GUIDANCE_SCALE,
            'sample_stride': SAMPLE_STRIDE,
            'fps': OUTPUT_FPS,
            'seed': SEED
        }
    ]
}

config_path = '/content/MimicMotion/configs/colab_run.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('✅ Config generada en:', config_path)
print(yaml.dump(config, default_flow_style=False))

In [ ]:
import os

os.chdir('/content/MimicMotion')
os.makedirs('outputs', exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:256'

# El subproceso de Python que ejecuta inference.py tiene su propio espacio de módulos,
# así que escribimos un wrapper que aplica el patch ANTES de que inference.py importe diffusers.
no_fp16_args = ",'--no_use_float16'" if not USE_FLOAT16 else ''

wrapper_code = f"""
# Patch: cached_download fue eliminado en huggingface_hub>=0.24
import huggingface_hub as _h
if not hasattr(_h, 'cached_download'):
    _h.cached_download = _h.hf_hub_download

import sys
sys.argv = ['inference.py',
            '--inference_config', '{config_path}',
            '--output_dir', 'outputs/'{no_fp16_args}]

# Ejecutar inference.py en el mismo proceso (patch ya activo)
with open('inference.py') as _f:
    exec(_f.read(), {{'__name__': '__main__', '__file__': 'inference.py'}})
"""

with open('/tmp/run_inference.py', 'w') as f:
    f.write(wrapper_code)

print('🚀 Ejecutando inferencia...')
print('─' * 60)
!PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:256 python /tmp/run_inference.py

## Paso 6 — Ver y descargar el resultado

In [ ]:
import glob, os
from IPython.display import HTML, display as ipy_display
from base64 import b64encode

# Buscar el video generado más reciente
output_videos = sorted(glob.glob('/content/MimicMotion/outputs/**/*.mp4', recursive=True),
                        key=os.path.getmtime, reverse=True)

if not output_videos:
    print('❌ No se encontraron videos de salida. Revisa los errores en la celda anterior.')
else:
    latest_video = output_videos[0]
    size_mb = os.path.getsize(latest_video) / 1e6
    print(f'✅ Video generado: {latest_video} ({size_mb:.1f} MB)')

    # Reproducir en el notebook
    with open(latest_video, 'rb') as f:
        video_data = f.read()
    b64 = b64encode(video_data).decode()
    html = f'''
    <video controls width="480">
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''
    ipy_display(HTML(html))

In [ ]:
# Descargar el video a tu computadora
from google.colab import files

if output_videos:
    print(f'⬇️  Descargando {os.path.basename(latest_video)}...')
    files.download(latest_video)
else:
    print('❌ No hay video para descargar.')

## (Opcional) Procesar múltiples pares foto/video en lote

In [ ]:
# ============================================================
# Procesamiento en lote: define tus pares aquí
# ============================================================

# Sube todos tus archivos primero con files.upload(), luego
# define la lista de pares (imagen, video) a procesar.

BATCH_CASES = [
    # {'ref_image_path': '/content/inputs/persona1.jpg', 'ref_video_path': '/content/inputs/dance1.mp4'},
    # {'ref_image_path': '/content/inputs/persona2.jpg', 'ref_video_path': '/content/inputs/dance2.mp4'},
]

if not BATCH_CASES:
    print('ℹ️  Agrega pares a BATCH_CASES para procesamiento en lote.')
else:
    import yaml, os
    os.chdir('/content/MimicMotion')

    test_cases = []
    for case in BATCH_CASES:
        test_cases.append({
            **case,
            'num_frames': NUM_FRAMES,
            'resolution': RESOLUTION,
            'frames_overlap': FRAMES_OVERLAP,
            'num_inference_steps': NUM_INFERENCE_STEPS,
            'noise_aug_strength': NOISE_AUG_STRENGTH,
            'guidance_scale': GUIDANCE_SCALE,
            'sample_stride': SAMPLE_STRIDE,
            'fps': OUTPUT_FPS,
            'seed': SEED
        })

    batch_config = {
        'base_model_path': 'stabilityai/stable-video-diffusion-img2vid-xt-1-1',
        'ckpt_path': 'models/MimicMotion_1-1.pth',
        'test_cases': test_cases
    }

    batch_config_path = '/content/MimicMotion/configs/colab_batch.yaml'
    with open(batch_config_path, 'w') as f:
        yaml.dump(batch_config, f, default_flow_style=False)

    print(f'🚀 Procesando {len(BATCH_CASES)} casos en lote...')
    !PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:256 python inference.py \
        --inference_config {batch_config_path} \
        --output_dir outputs/batch/

---
## Referencia de parámetros

| Parámetro | Descripción | Valor típico |
|---|---|---|
| `resolution` | Alto de salida en píxeles | 512–576 |
| `num_frames` | Frames totales a generar | 16–72 |
| `sample_stride` | Intervalo de muestreo del video ref. | 1–4 |
| `frames_overlap` | Frames de solapamiento entre segmentos | 4–12 |
| `num_inference_steps` | Pasos del modelo de difusión | 20–30 |
| `guidance_scale` | Fuerza de seguimiento de la pose | 1.5–3.0 |
| `noise_aug_strength` | Variación adicional (0 = sin ruido) | 0.0 |
| `fps` | FPS del video de salida | 15 |
| `seed` | Semilla para reproducibilidad | 42 |

### Consejos para mejor calidad
- La **foto del personaje** debe ser de cuerpo completo, fondo limpio, buena iluminación.
- El **video de referencia** debe mostrar el cuerpo completo con movimientos claros.
- Si hay OOM (out of memory): reduce `resolution` a 512 o `num_frames` a 16–32.
- Para videos más largos usa `sample_stride=2` y procesa en segmentos.